# Chapter 9 — Conditional Independence and d-Separation

Companion notebook for the [probintro textbook](https://josephausterweil.github.io/probintro/intro2/09_conditional_independence/).

Build the rain / sprinkler / wet-floor **collider** and watch **explaining away** happen in code.

## Setup

Run this first (on Colab, restart the runtime afterward if prompted).

In [ ]:
!pip install genjax -q
print('✅ ready')

## The collider: rain → wet ← sprinkler

Two independent causes (each prior 0.3) of one effect, via a deterministic OR.

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
from genjax import gen, flip, ChoiceMap

@gen
def wet_floor():
    rain = flip(0.3) @ "rain"
    sprinkler = flip(0.3) @ "sprinkler"
    either = jnp.maximum(rain.astype(int), sprinkler.astype(int))
    p_wet = either.astype(float)          # 1.0 if either cause fired, else 0.0
    wet = flip(p_wet) @ "wet"
    return rain, sprinkler, wet

def posterior_rain(constraints, n=40000, seed=0):
    """P(rain = 1 | constraints) by importance sampling."""
    keys = jr.split(jr.key(seed), n)
    def one(k):
        trace, log_weight = wet_floor.generate(k, constraints, ())
        return trace.get_choices()["rain"].astype(float), log_weight
    rain_samples, log_weights = jax.vmap(one)(keys)
    weights = jnp.exp(log_weights - jnp.max(log_weights))
    weights = weights / jnp.sum(weights)
    return float(jnp.sum(rain_samples * weights))

## Explaining away, step by step

Watch belief in rain rise when the floor is wet, then fall back to its prior once the sprinkler explains the wetness.

In [ ]:
print(f"P(rain)                       = 0.300   (prior)")
print(f"P(rain | wet)                 = {posterior_rain(ChoiceMap.d({'wet': 1})):.3f}")
print(f"P(rain | wet, sprinkler on)   = {posterior_rain(ChoiceMap.d({'wet': 1, 'sprinkler': 1})):.3f}")
print(f"P(rain | wet, sprinkler off)  = {posterior_rain(ChoiceMap.d({'wet': 1, 'sprinkler': 0})):.3f}")

## Your turn

1. **d-separation:** estimate $P(\text{rain} \mid \text{sprinkler}=1)$ *without* observing the wet floor. Is it the prior 0.3? (Rain and sprinkler are independent until you condition on their shared effect.)
2. **Noisy-OR:** change `p_wet` so each active cause wets the floor with probability 0.9 instead of certainty. Does rain return *exactly* to its prior after the sprinkler, or stay a little above?
3. Try the chain and fork patterns as their own little models and confirm that conditioning on the middle node *blocks* them.